# 02 - Analyze

Copyright (c) Microsoft Corporation. Licensed under the MIT License.

Reads the raw JSON from the Collect stage (same `RUN_ID` folder in the Lakehouse), evaluates configured analyzers against the checklist, and merges to `findings.json`. No Azure/Fabric auth is needed - analyzers are offline. Notebook analysis preserves authoritative workspace/notebook IDs and groups identified hits by IDs; it never reconstructs missing provenance from names for the optional owner projection. Mixed central findings are not copied into the owner report.


## Parameters


In [ ]:
# Parameters (injected by the pipeline)
CLIENT_NAME = "Contoso"
ENGAGEMENT_NAME = "Fabric Architecture Review"
REVIEWER_NAME = ""
WORKSPACE_IDS = ""  # same scope as 01 Collect; evidence is bound to it
RUN_ID = "latest"

# Run configuration bindings populated by setup.
FAR_THRESHOLDS_LIBRARY = "FAR_Thresholds"
FAR_CONNECTIONS_LIBRARY = "FAR_Connections"


## Load staged FAR runtime


In [ ]:
import os, sys, runpy
from pathlib import Path
import notebookutils
ctx = notebookutils.runtime.context
if (ctx.get("defaultLakehouseName") != "FOS_Lakehouse" or not ctx.get("defaultLakehouseId")
        or ctx.get("defaultLakehouseWorkspaceId") != (ctx.get("currentWorkspaceId") or ctx.get("workspaceId"))):
    raise ValueError("Attach this workspace's FOS_Lakehouse as default and restart the session.")
REPO_DIR = "/lakehouse/default/Files/far/runtime/source"
if not Path(REPO_DIR, "fabric", "bootstrap.py").is_file():
    raise FileNotFoundError("FAR runtime not staged. Deploy the architecture-review module before running.")
runpy.run_path(str(Path(REPO_DIR, "fabric", "bootstrap.py")))


## Analyze + merge findings


In [ ]:
import json, os, subprocess, sys
from orchestration.notebook_parameters import optional_string
os.environ["CLIENT_NAME"] = optional_string("CLIENT_NAME", CLIENT_NAME)
os.environ["ENGAGEMENT_NAME"] = optional_string("ENGAGEMENT_NAME", ENGAGEMENT_NAME)
os.environ["REVIEWER_NAME"] = optional_string("REVIEWER_NAME", REVIEWER_NAME)
os.environ["WORKSPACE_IDS"] = optional_string("WORKSPACE_IDS", WORKSPACE_IDS)

from orchestration.storage import latest_run_id, run_folder
if RUN_ID == "latest":
    RUN_ID = latest_run_id()
os.environ["RUN_ID"] = RUN_ID
OUT_DIR = str(run_folder(RUN_ID))
RAW_DIR = os.path.join(OUT_DIR, "raw")
from pathlib import Path
from orchestration.review_config import activate
activate(Path(OUT_DIR) / "review_config.json")
import notebookutils
from orchestration.review_workspace import export_review_workspace
export_review_workspace(notebookutils)
if not os.path.isdir(RAW_DIR):
    raise RuntimeError("Raw folder not found: " + RAW_DIR + " -- run the Collect stage first.")
os.environ["OUTPUT_DIR"] = OUT_DIR

# Never let a failed run merge findings or a completion manifest left by an earlier attempt.
for _name in os.listdir(OUT_DIR):
    if _name in ("findings.json", "run_manifest.json") or (_name.startswith("findings_") and _name.endswith(".json")):
        os.remove(os.path.join(OUT_DIR, _name))

_failures = []
def _run(mod, args):
    print("\n==>", mod)
    r = subprocess.run([sys.executable, "-m", mod, *args], cwd=REPO_DIR)
    if r.returncode:
        _failures.append({"module": mod, "returncode": r.returncode})
        print("  ", mod, "exit", r.returncode)

_outputs = []
for mod, fn in [
    ("analyzers.far_review", "findings_architecture.json"),
]:
    _outputs.append(fn)
    _run(mod, ["--raw-dir", RAW_DIR, "--out-dir", OUT_DIR])

_invalid = []
for _name in _outputs:
    _path = os.path.join(OUT_DIR, _name)
    try:
        with open(_path, "r", encoding="utf-8-sig") as _handle:
            _payload = json.load(_handle)
        if not isinstance(_payload, list) or not _payload:
            _invalid.append(_name + " is empty or not a finding list")
    except Exception as _error:
        _invalid.append(_name + ": " + str(_error))
if _failures or _invalid:
    raise RuntimeError("Analysis incomplete: failures=" + json.dumps(_failures) + "; invalid_outputs=" + json.dumps(_invalid))

_run("analyzers.merge_findings", ["--out-dir", OUT_DIR, "--run-id", RUN_ID])
if _failures:
    raise RuntimeError("Finding merge failed: " + json.dumps(_failures))

with open(os.path.join(OUT_DIR, "findings.json"), "r", encoding="utf-8-sig") as _handle:
    _merged = json.load(_handle)
_expected_count = sum(len(json.load(open(os.path.join(OUT_DIR, _name), "r", encoding="utf-8-sig"))) for _name in _outputs)
if not isinstance(_merged, list) or len(_merged) != _expected_count:
    raise RuntimeError("Merged findings completeness check failed")
with open(os.path.join(OUT_DIR, "run_manifest.json"), "r", encoding="utf-8-sig") as _handle:
    _manifest = json.load(_handle)
if _manifest.get("run_id") != RUN_ID or _manifest.get("finding_count") != len(_merged) or _manifest.get("status") != "completed":
    raise RuntimeError("Run manifest completeness check failed")

import notebookutils
notebookutils.notebook.exit("analyzed:" + OUT_DIR + " | findings=" + str(len(_merged)))
